# Resume Matching

## Load Libraries

In [ ]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

## Load the dataset

In [ ]:
resume_df = pd.read_csv("/content/gdrive/My Drive/Colab Notebooks/Final Capstone Project /resume_clean.csv")
job_df = pd.read_csv("/content/gdrive/My Drive/Colab Notebooks/Final Capstone Project /job_clean.csv")

print("Resume Dataset", resume_df.shape)
print("Job Dataset", job_df.shape)


In [ ]:
resume_df.head()

In [ ]:
job_df.head()

In [ ]:
resume_df.columns

In [ ]:
job_df.columns

In [ ]:
##Preparing the text - replacing the missing value with empty string

resume_df['clean_description'] = resume_df['clean_description'].fillna("")
job_df['clean_description'] = job_df['clean_description'].fillna("")


In [ ]:
#Choosing only 1 resume

resume_index = 0
resume_text = resume_df.loc[resume_index]['clean_description']
print(resume_text)

In [ ]:
#Combining resume and job description

combined_text = [resume_text]  + job_df['clean_description'].tolist()
print(combined_text)

## TF-IDF Vectorization - Baseline Model

### TF-IDF Vectorization

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

#Intitialize and run the vectorizer
vectorizer = TfidfVectorizer(stop_words = 'english') #stopwords will remove the common english words
tfidf_matrix = vectorizer.fit_transform(combined_text)
print(tfidf_matrix.shape)

### Cosine Similarity

In [ ]:
#calculating the cosine similarity
from sklearn.metrics.pairwise import cosine_similarity

similarity_scores = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:]).flatten()
print(similarity_scores.shape)
print(similarity_scores[:5])

In [ ]:
#Attaching scores to a result table
results = job_df[["company_name", "position_title", "Required Skills"]].copy()
results["similarity_scores"] = similarity_scores

### Top Job Recommendations

In [ ]:
#Top 5 matches
top_5_jobs = results.sort_values(by = "similarity_scores", ascending = False).head(5)
print(top_5_jobs)


In [ ]:
top_5_jobs = top_5_jobs.copy()
top_5_jobs["match_percentage"] = (top_5_jobs["similarity_scores"] * 100).round(2)
top_5_jobs[["company_name", "position_title", "match_percentage"]]

In [ ]:
#top_5_jobs[["position_title", "Required Skills"]]

Based on the results, the TF-IDF model provides a simple baseline for matching resumes with job descriptions. It compares the words that appear in both documents, but it does not understand the meaning or context of the text. Because of this, it may recommend jobs from related fields instead of the candidate's main area of expertise. Therefore, while TF-IDF is useful for initial resume matching, it has limitations in producing highly accurate job recommendations.

### Similarity Analysis

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

resume_text = resume_df["Resume_str"].fillna("")
job_text = job_df["job_description"].fillna("")

all_text = pd.concat([resume_text, job_text], ignore_index=True)

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

X = vectorizer.fit_transform(all_text)

# split
resume_vectors = X[:len(resume_df)]
job_vectors = X[len(resume_df):]

# sim
similarity = cosine_similarity(resume_vectors, job_vectors)

In [ ]:
similarity.shape

In [ ]:
import plotly.express as px
sample = similarity[:50, :50]

fig = px.imshow(
    sample,
    title="Resume-Job Similarity Matrix"
)

fig.update_layout(
    width=850,
    height=700
)

fig.show()

In [ ]:
best_similarity = similarity.max(axis=1)

fig = px.violin(
    y=best_similarity,
    box=True,
    points=False,
    title="Distribution Resume–Job Similarity Scores",
    labels={"y": "Cosine Similarity"}
)

fig.show()

## Future Work

1. Sentence Transformer - Explore semantic embeddings to improve resume-job matching by capturing contextual meaning rather than relying primarily on keyword overlap.
2. Skill Gap Analysis - After identifying suitable job matches, compare candidate skills with job requirements to identify missing skills.